# Supply Chain Twin: Google Colab notebook (data on Google Drive)

Generated by `scripts/build_notebook.py` (at commit `d7ec4b4`). The setup cell checks out the
code from GitHub (`REF`, default `main`) into `/content/SupplyTwinAI` and installs it, so Colab
runs exactly the pushed code; the checked-out commit is printed. To change the code, push to
GitHub (or set `REF` to another branch, tag or commit) and re-run the setup cell.

**Google Drive layout** (created on first run; change `DRIVE` in the setup cell to move it):

```
MyDrive/SupplyTwinAI/
  data/raw/provenance.json          dataset (pinned revision), about 110 MB
  data/raw/data/train.parquet
  data/raw/data/eval.parquet        locked final test: integrity-checked, never loaded
  runs/                             all run outputs (metrics, predictions, posteriors)
```

The dataset is read from Drive and copied to the Colab disk for fast reads. If it is not on
Drive yet, the pinned revision is downloaded once and saved there. You can also upload a
local `data/raw/` folder (with `provenance.json`) to that place yourself.

**How to use:** run *Setup*, *Helpers* and *Data* (Runtime → Run all is fine), then the steps
you need. Finished steps keep their outputs on Drive, so you can skip them later.
`FAST = True` (in the helpers cell) keeps every step to a few minutes; set `FAST = False` for
the full runs. If Colab restarts the session, re-run *Setup* and *Helpers* first.
Use a CPU runtime with many cores: the installed JAX is CPU-only, so a GPU is not used.

Rules that every step follows are in `CLAUDE.md` of the repository: rolling-origin
backtesting without leakage, point-in-time features, the official `eval` split is never read.
Sales are normalized units; reconstructed demand is never presented as observed.

## Setup

In [1]:
import os, sys, pathlib, shutil, subprocess
from google.colab import drive

REPO = "https://github.com/moghbaie/SupplyTwinAI.git"
REF = "main"  # branch, tag or commit to run
DRIVE = pathlib.Path("/content/drive/MyDrive/SupplyTwinAI")  # dataset and run outputs live here
ROOT = pathlib.Path("/content/SupplyTwinAI")  # code checkout (Colab disk, wiped per runtime)

def sh(cmd):
    """Run a shell command in ROOT; stop the cell if it fails."""
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)

drive.mount("/content/drive")
for d in (DRIVE / "data" / "raw", DRIVE / "runs", ROOT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# Check out the code (works in an empty folder and over leftovers of an older notebook;
# untracked files such as data/ and the runs link are kept, tracked files are overwritten).
if not (ROOT / ".git").exists():
    sh(f"git init -q && git remote add origin {REPO}")
sh(f"git fetch -q --depth 1 origin {REF} && git checkout -q --force FETCH_HEAD")
sh('git log -1 --format="code: commit %h (%cd) %s" --date=short')

if not os.path.islink("runs"):  # run outputs go straight to Drive
    if os.path.isdir("runs"):
        os.rmdir("runs")  # only succeeds if empty, so nothing is lost
    os.symlink(DRIVE / "runs", "runs")

# Install with all extras (no --force-reinstall: it would upgrade Colab's numpy and force a
# session restart). The import check runs in a fresh process, like the run cells below.
sh('pip install -q -e ".[dev,models,bayes]"')
sh('python -c "import sctwin.censoring, sctwin.demand_model; print(\'package import OK\')"')
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # make the package importable in this kernel too
print("code:", ROOT, "| data and runs:", DRIVE, "| CPU cores:", os.cpu_count())

Mounted at /content/drive
code: /content/SupplyTwinAI | data and runs: /content/drive/MyDrive/SupplyTwinAI | CPU cores: 2


## Helpers

In [2]:
import copy, glob, json
import pandas as pd
import yaml

os.chdir(ROOT)

FAST = True          # small, quick runs; set False for the full runs
CORES = os.cpu_count() or 2

def config_with(path, updates, name):
    """Copy a YAML config with nested updates applied; returns the new path."""
    cfg = yaml.safe_load(open(path, encoding="utf-8"))
    def merge(a, b):
        for k, v in b.items():
            if isinstance(v, dict) and isinstance(a.get(k), dict):
                merge(a[k], v)
            else:
                a[k] = v
    merge(cfg, copy.deepcopy(updates))
    out = f"configs/nb_{name}.yaml"
    yaml.safe_dump(cfg, open(out, "w", encoding="utf-8"), sort_keys=False)
    return out

def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(pattern)
    print("reading", files[-1])
    return json.load(open(files[-1], encoding="utf-8"))

def model_table(models):
    rows = {}
    for m, r in models.items():
        iv = r.get("intervals", {})
        rows[m] = {"mae": r.get("mae", r.get("mae_median")), "bias": r.get("bias"),
                   "wis": r.get("wis"), "cov50": iv.get("0.5", {}).get("coverage"),
                   "cov90": iv.get("0.9", {}).get("coverage")}
    return pd.DataFrame(rows).T.round(4)

## Run: tests (synthetic, no data needed)
The leakage tests run for both units, "day" and "hour". Warnings are errors, except
deprecation warnings: Colab's preinstalled libraries raise them inside each other (e.g.
matplotlib calling deprecated pyparsing), which is not this project's code. Local runs and CI
keep plain `-W error`.

In [ ]:
!python -m pytest -W error -W "default::DeprecationWarning" -m "not data" -q -p no:cacheprovider

## Data from Google Drive (run before any step)
Uses the dataset on Drive if present (checksums and pinned revision verified); otherwise
downloads the pinned revision once and saves it to Drive. Copies it to the Colab disk.

In [6]:
import json, yaml
from sctwin.data import verify_checksums

os.chdir(ROOT)
raw_drive = DRIVE / "data" / "raw"
pinned = yaml.safe_load(open("configs/small.yaml"))["dataset"]["revision"]
skip_cache = shutil.ignore_patterns(".cache")
if (raw_drive / "provenance.json").exists():
    print("dataset found on Drive:", raw_drive)
    shutil.copytree(raw_drive, "data/raw", dirs_exist_ok=True, ignore=skip_cache)
else:
    print("dataset not on Drive yet: downloading the pinned revision once")
    !python scripts/fetch_data.py
    shutil.copytree("data/raw", raw_drive, dirs_exist_ok=True, ignore=skip_cache)
    print("saved to Drive:", raw_drive)
prov = json.load(open("data/raw/provenance.json"))
assert prov["revision"] == pinned, f"Drive has revision {prov['revision']}, config pins {pinned}"
checks = verify_checksums("data/raw")
assert all(checks.values()), f"checksum mismatch: {checks}"
print("revision", pinned[:12], "checksums OK:", checks)

dataset found on Drive: /content/drive/MyDrive/SupplyTwinAI/data/raw
revision c374e85975c8 checksums OK: {'train': True, 'eval': True}


## Run: Step 1, audit and real-data tests
Re-runs the audit (it re-writes the identical split manifest) and the real-data tests,
including the real-data leakage checks.

In [7]:
!python scripts/audit_data.py
!python -m pytest -m data -q -p no:cacheprovider

{
  "gates": {
    "train.schema_ok": true,
    "train.no_duplicate_keys": true,
    "train.dates_parse": true,
    "train.arrays_len_24": true,
    "train.no_negative_or_nan_sales": true,
    "train.status_binary": true,
    "eval.schema_ok": true,
    "eval.no_duplicate_keys": true,
    "eval.dates_parse": true,
    "eval.arrays_len_24": true,
    "eval.no_negative_or_nan_sales": true,
    "eval.status_binary": true,
    "train.hierarchy_strict_tree": true,
    "cross.no_overlapping_keys": true,
    "cross.test_strictly_after_train": true
  },
  "all_gates_pass": true,
  "series_counts": {
    "total": 50000,
    "in_sample": 43883,
    "holdout_store_only": 4758,
    "holdout_product_only": 1242,
    "holdout_both": 117
  },
  "runtime_s": 94.0
}
......                                                                   [100%]


## Run: Step 3, baseline backtest (observed sales)
Rolling origin; train on feature times t <= T - L - H; predict (T + L, T + L + H].
Full day run: about 15 min on 12 cores and about 6 GB RAM. FAST uses 200 series and 2 folds.

In [ ]:
cfg3 = config_with("configs/baselines.yaml", {"lgbm": {"num_threads": CORES}}, "baselines")
smoke = "--smoke" if FAST else ""
!python scripts/run_baselines.py --unit day --config {cfg3} {smoke}
!python scripts/run_baselines.py --unit hour --config {cfg3} {smoke}

In [ ]:
for unit in ("day", "hour"):
    m = latest(f"runs/step3*/{unit}/*/metrics.json")
    print(unit, "origins", m["protocol"]["origins"][:5], "...")
    display(model_table(m["metrics"]["overall"]["models"]))
    display(pd.DataFrame({o: {k: v.get("wis", v.get("mae")) for k, v in f["models"].items()}
                          for o, f in m["metrics"]["folds"].items()}).T.round(4))

## Run: Step 4, censoring benchmark (proxy truth)
Demand-driven stockouts injected into fully in-stock days; calibration on the fit window only.
Full run: about 1.5 min for 5,000 series. FAST uses 300 series.

In [ ]:
smoke = "--smoke" if FAST else ""
!python scripts/run_censoring_benchmark.py {smoke}

In [ ]:
m = latest("runs/step4*/*/metrics.json")
print("calibrated injection:", m["calibrated_config"])
display(pd.DataFrame({"real": m["real_fit_window"]["stats"],
                      "injected": m["injected"]["stats"]}).round(3))
display(pd.DataFrame({"real": m["real_fit_window"]["position_ratios"],
                      "injected": m["injected"]["position_ratios"]}).round(3))
print("daily totals on stockout days")
display(model_table(m["benchmark"]["daily"]["overall"]["models"]))

## Run: Step 5, Bayesian model ladder (M0, M1, M2, M1d, M2d)
NumPyro NUTS. FAST: 4 series and short chains (a few minutes). Full: 20 series, 4 chains x
(1000 + 1000) draws; chains run in parallel, one per CPU core (up to 4), so it takes hours on a
2-core runtime. `--part real` and `--part recovery` split the run in two.
M3 = M2 with `--n-series 100 --models M2`. M1d/M2d were added after the first run and declared
primary on 2026-11-17 (plan.md, "Step 5 revision"); earlier results for them are exploratory.

In [ ]:
chains = min(4, CORES)
cfg5 = config_with("configs/bayes.yaml", {"mcmc": {"num_chains": chains}}, "bayes")
smoke = "--smoke" if FAST else ""
!python scripts/run_bayes.py --config {cfg5} --part all {smoke}

In [ ]:
m = latest("runs/step5*/*/metrics_all.json")
real = m["real"]
keys = ("runtime_s", "divergences", "max_rhat", "min_ess")
display(pd.DataFrame({k: {x: v["diagnostics"][x] for x in keys}
                      for k, v in real["models"].items()}).T)
print("daily totals on stockout days (proxy truth); baselines are point estimates")
display(model_table(real["daily"]["overall"]["models"]))
print("PSIS-LOO on in-stock hours:", real["loo_compare"])
print("paired cluster bootstrap (clusters = series), Holm-adjusted; "
      "mean_diff < 0 = lower loss than the reference")
for name, c in real["comparisons"].items():
    print(name)
    display(pd.DataFrame({k: {"mean_diff": v["mean_diff"], "ci_low": v["ci"][0],
                              "ci_high": v["ci"][1], "relative": v["relative"],
                              "p_holm": v["p_holm"]}
                          for k, v in c["comparisons"].items()}).T.round(4))
for name, r in m["recovery"]["models"].items():
    print("recovery", name, r["mean_window_demand"])
    display(pd.DataFrame(r["scalars"]).T)

## Run: Step 5 revision, replicated level recovery (flagged-hour treatment)
Pre-declared check (plan.md, "Step 5 revision"): with flagged hours dropped (M1d), the demand
level must be within ±1% of a paired fit on the uncensored demand, with the 95% interval
including 0. Static truth, 8 replications of oracle, M0, M1, M1d (4 chains x 500 + 500).
Replications run in parallel, one worker per 4 cores. The second run is the declared follow-up:
the same check with an opening stock that does not scale with the true day effect, which
isolates that channel (oracle and M1d only). Together about 2-3 h on 12 cores.
FAST: 2 replications, 4 series, short chains. Partial results are saved after each replication.

In [8]:
chains = min(4, CORES)
smoke = "--smoke" if FAST else ""
follow_up = "--stock no_day_effect --models oracle,M1d"
!python scripts/run_level_recovery.py --chains {chains} {smoke}
!python scripts/run_level_recovery.py --chains {chains} {follow_up} {smoke}

2 replications x ['oracle', 'M0', 'M1', 'M1d'] on 4 series x 76 days, stock day_effect; 1 workers x 2 chains; out runs/level_recovery/acece47200a6
rep 0: oracle -3.73%  M0 -9.65%  M1 -1.64%  M1d -3.69%  (750 s)
rep 1: oracle +4.03%  M0 +0.25%  M1 +10.30%  M1d +8.13%  (1186 s)
{
 "n_reps": 2,
 "oracle": {
  "rel_error_vs_truth": 0.001491322224077385,
  "coverage90": 0.0
 },
 "M0": {
  "rel_error_vs_truth": -0.047042245347106826,
  "coverage90": 0.5,
  "rel_diff_vs_oracle": {
   "mean": -0.04894835790884411,
   "ci95": [
    -0.20866856534069372,
    0.1107718495230055
   ]
  }
 },
 "M1": {
  "rel_error_vs_truth": 0.04329101302822719,
  "coverage90": 0.5,
  "rel_diff_vs_oracle": {
   "mean": 0.040989074494612976,
   "ci95": [
    -0.2044216654330303,
    0.28639981442225626
   ]
  }
 },
 "M1d": {
  "rel_error_vs_truth": 0.02224203217523557,
  "coverage90": 0.5,
  "rel_diff_vs_oracle": {
   "mean": 0.01996474134366244,
   "ci95": [
    -0.22764191132557005,
    0.2675713940128949
   ]
  }

In [9]:
for path in sorted(glob.glob("runs/level_recovery/*/results.json"), key=os.path.getmtime)[-2:]:
    r = json.load(open(path, encoding="utf-8"))
    s = r["summary"]
    print(f"\n{path}: stock = {r['settings']['stock']}, {s['n_reps']} replications, "
          f"smoke = {r['settings']['smoke']}")
    display(pd.DataFrame({m: {"rel_error_vs_truth": v["rel_error_vs_truth"],
                              "coverage90": v["coverage90"],
                              "rel_diff_vs_oracle": v.get("rel_diff_vs_oracle", {}).get("mean"),
                              "ci95": v.get("rel_diff_vs_oracle", {}).get("ci95")}
                          for m, v in s.items() if isinstance(v, dict) and "coverage90" in v}).T)
    print("acceptance (M1d, declared configuration only):", s.get("acceptance_M1d"))
    display(pd.DataFrame({(x["rep"], m): {k: v[k] for k in ("runtime_s", "divergences", "max_rhat")}
                          for x in r["replications"] for m, v in x["models"].items()}).T)


runs/level_recovery/acece47200a6/results.json: stock = day_effect, 2 replications, smoke = True


,rel_error_vs_truth,coverage90,rel_diff_vs_oracle,ci95
oracle,0.001491,0.0,NaN,NaN
M0,-0.047042,0.5,-0.048948,"[-0.20866856534069372, 0.1107718495230055]"
M1,0.043291,0.5,0.040989,"[-0.2044216654330303, 0.28639981442225626]"
M1d,0.022242,0.5,0.019965,"[-0.22764191132557005, 0.2675713940128949]"


acceptance (M1d, declared configuration only): {'rule': 'mean relative difference vs oracle within +-1% and its 95% CI includes 0 (plan.md, Step 5 revision, declared 2026-11-17)', 'passed': False}


runtime_s  divergences  max_rhat
0 oracle      238.8          0.0  1.183513
  M0          295.2         12.0  1.163778
  M1          111.0          0.0  1.198673
  M1d         105.0          0.0  1.227919
1 oracle      125.0          0.0  1.220655
  M0          111.9          0.0  1.168755
  M1          104.8          0.0  1.249660
  M1d          94.3          0.0  1.146210


runs/level_recovery/1c99b2c45356/results.json: stock = no_day_effect, 2 replications, smoke = True


,rel_error_vs_truth,coverage90,rel_diff_vs_oracle,ci95
oracle,0.001491,0.0,NaN,NaN
M1d,-0.006737,0.5,-0.008584,"[-0.12924028067848572, 0.11207198343197725]"


acceptance (M1d, declared configuration only): None


runtime_s  divergences  max_rhat
0 oracle      237.6          0.0  1.183513
  M1d         237.1          0.0  1.060441
1 oracle      123.4          0.0  1.220655
  M1d         110.4          0.0  1.170353

## Run: Step 5, simulation-based calibration of M1
Ranks of the true parameters among posterior draws should be uniform (chi-square p-values not
small after Holm). Full: 200 replications (about 40 min). FAST: 20.

In [10]:
reps = 20 if FAST else 200
!python scripts/run_sbc.py --reps {reps}
s = latest("runs/sbc/*/ranks.json")
display(pd.DataFrame(s["scalars"]).T)
print("min Holm-adjusted p:", s["min_p_holm"])
print("divergent replications:", s["divergent_reps"])

rep 10/20  419 s
rep 20/20  665 s
  alpha    chi2 p=0.740 mean rank=0.41 [4, 1, 2, 3, 2, 3, 2, 0, 1, 2]
  sigma_a  chi2 p=0.163 mean rank=0.38 [5, 4, 1, 0, 3, 1, 3, 0, 2, 1]
  tau_g    chi2 p=0.213 mean rank=0.40 [3, 4, 5, 1, 0, 0, 2, 2, 1, 2]
  sigma_u  chi2 p=0.067 mean rank=0.52 [0, 1, 1, 4, 2, 5, 3, 4, 0, 0]
  b_disc   chi2 p=0.437 mean rank=0.49 [1, 4, 0, 4, 1, 3, 1, 2, 3, 1]
  b_act    chi2 p=0.276 mean rank=0.52 [0, 3, 1, 1, 4, 3, 4, 3, 0, 1]
  sigma_y  chi2 p=0.213 mean rank=0.40 [2, 5, 2, 3, 0, 4, 1, 0, 2, 1]
  k0       chi2 p=0.637 mean rank=0.44 [2, 2, 3, 3, 1, 1, 4, 3, 1, 0]
  k1       chi2 p=0.122 mean rank=0.63 [2, 0, 2, 2, 1, 0, 1, 5, 2, 5]
min Holm-adjusted p = 0.602; written to runs/sbc/20a4d182c47b
reading runs/sbc/20a4d182c47b/ranks.json


,hist,chi2_p,mean_rank_frac
alpha,"[4, 1, 2, 3, 2, 3, 2, 0, 1, 2]",0.739918,0.415
sigma_a,"[5, 4, 1, 0, 3, 1, 3, 0, 2, 1]",0.162606,0.3805
tau_g,"[3, 4, 5, 1, 0, 0, 2, 2, 1, 2]",0.213309,0.3955
sigma_u,"[0, 1, 1, 4, 2, 5, 3, 4, 0, 0]",0.066882,0.515
b_disc,"[1, 4, 0, 4, 1, 3, 1, 2, 3, 1]",0.437274,0.488
b_act,"[0, 3, 1, 1, 4, 3, 4, 3, 0, 1]",0.275709,0.5165
sigma_y,"[2, 5, 2, 3, 0, 4, 1, 0, 2, 1]",0.213309,0.403
k0,"[2, 2, 3, 3, 1, 1, 4, 3, 1, 0]",0.637119,0.4435
k1,"[2, 0, 2, 2, 1, 0, 1, 5, 2, 5]",0.122325,0.6335


min Holm-adjusted p: 0.6019342899671396
divergent replications: 3
